In [ ]:
# ════════════════════════════════════════════════════════════════════
# Stage 1a Pass 3 — Type Classification LoRA Adapter
# ════════════════════════════════════════════════════════════════════
# PIPELINE STAGES:
#   Stage 1a = Relevance, Priority, Type (3 LoRA passes per email)
#   Stage 1b = Identifying ideal vs actual context and evidence
#   Stage 2a = Complexity and urgency
#   Stage 2b = Research, recommendations and strategy
#
# THIS NOTEBOOK: Trains the Type LoRA (Pass 3 of Stage 1a).
#   Pass 1 — Relevance: is it relevant? (4 labels, 11/11 deployed)
#   Pass 2 — Priority: how urgent? (3 labels, 11/11 deployed)
#   Pass 3 — Type: what kind of email? (8 labels — THIS NOTEBOOK)
#
# Model: @cf/mistralai/mistral-7b-instruct-v0.2-lora
# Labels: legal, financial, regulatory, commercial, technical,
#         environmental, health, administrative
# Adapter worker: servers/mcp/network-http/cloudflare/ai/cf-adapters/
# Email worker: servers/mcp/network-http/cloudflare/email/cf-email/
# Checklist: notebooks/checklists/stage-1a-filter/type-relevance-priority.json
# Schema: schemas/mobicycle-us/pipelines/email/email_enrichment.json
# Output fields → enrichment schema: type_class, type_confidence, type_model, type_adapter_id
# Run once per account. Change ACCOUNT below.
#
# RULES (from notebooks/platforms/errors-do-not-delete/):
#   1. NEVER delete or overwrite existing comments — especially Cell 3 descriptions.
#   2. Before editing, verify every cell achieves the purpose stated above.
#   3. Colab MCP tools may not register. Fall back to editing .ipynb files locally.
#   4. Do NOT call open_colab_browser_connection if a notebook is already open.
#
# WARNING: This notebook gets overwritten when regenerated.
# Once trained and verified, push to GitHub immediately:
#   Training repos: org-mobicycle/training, org-mobicycle-ee/training, etc.
#   NEVER push to MobiCycleLtd/technologies.
# ════════════════════════════════════════════════════════════════════
# ── Cell 0: Config ──
ACCOUNT = "mobicycle.us"
TASK = "type"
ADAPTER_PREFIX = "type"
LABELS = ["legal", "financial", "regulatory", "commercial", "technical", "environmental", "health", "administrative"]
MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
MODEL_TYPE = "mistral"
WORKERS_AI_MODEL = "@cf/mistralai/mistral-7b-instruct-v0.2-lora"
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
BLOCK_SIZE = 512
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
GRADIENT_ACCUMULATION = 4
import json
_cfg = {k: v for k, v in locals().items() if k.isupper()}
with open('/content/config.json', 'w') as f:
    json.dump(_cfg, f)
print(f'Account: {ACCOUNT}')
print(f'Task: {TASK}')
print(f'Labels: {LABELS}')
print(f'Model: {MODEL}')
print('Config saved.')

In [ ]:
# ── Cell 1: Install ──
!pip install -q torch transformers accelerate peft bitsandbytes datasets trl


In [ ]:
# ── Cell 2: GPU Check ──
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu} ({mem:.1f} GB)")
print(f"CUDA: {torch.version.cuda}")


In [ ]:
# ════════════════════════════════════════════════════════════════════
# TYPE CLASSIFICATION — 8 CLASSES
# ════════════════════════════════════════════════════════════════════
# The LoRA evaluates emails with common sense and assigns a type.
# These are guiding tendencies, NOT rules. The AI develops its own
# understanding of what each type means for each account.
#
# LEGAL:
#   Court filings, solicitor correspondence, litigation
#   Contract disputes, enforcement actions, tribunal proceedings
#   Settlement discussions, witness statements, judicial reviews
#
# FINANCIAL:
#   Invoices, payments, tax correspondence (HMRC, Revenue, EMTA)
#   Banking, insurance, pensions, payroll, financial reporting
#   Expense claims, budget approvals, audit queries
#
# REGULATORY:
#   Government regulators, permits, licences, compliance reporting
#   Inspections, standards bodies, accreditation, registrations
#   Companies House filings, data protection (ICO), Ofcom, FCA
#
# COMMERCIAL:
#   Client proposals, supplier negotiations, partnership discussions
#   RFPs, tenders, sales enquiries, contract renewals, SoWs
#   Business development, pricing, delivery coordination
#
# TECHNICAL:
#   Cloud infrastructure, deployments, security alerts, monitoring
#   GitHub, Cloudflare, DNS, SSL, API changes, outage notifications
#   Software updates, vulnerability patches, system migrations
#
# ENVIRONMENTAL:
#   ESG reporting, emissions data, carbon targets (SBTi, CDP)
#   Waste management, WEEE compliance, biodiversity assessments
#   Circular economy, CSRD, sustainability frameworks, GRI
#
# HEALTH:
#   Workplace H&S, incident reports, risk assessments, RIDDOR
#   Product safety, COSHH, occupational health, PPE
#   HSE inspections, fire safety, wellbeing programmes
#
# ADMINISTRATIVE:
#   HR, recruitment, office management, travel, internal comms
#   Leave requests, performance reviews, training, onboarding
#   Facilities, scheduling, employee relations, payroll admin
#
# NOTE: Some emails span multiple types. The model picks the PRIMARY type.
# Type determines which complexity rubric Stage 2 applies.
# ════════════════════════════════════════════════════════════════════
# ── Cell 3: Training Data (Black-and-White Examples) ──
# 160 examples across 8 type classes. The model generalizes to gray areas.
import csv, os, random, json

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

# Format: (label, from_name, from_email, subject)
EXAMPLES = [
    # ──── LEGAL (20) ────
    ("legal", "Crown Court Listing", "listing@courts.gsi.gov.uk", "Hearing listed: Case MC-2025-0047"),
    ("legal", "Opposing Counsel", "litigation@lawfirm.co.uk", "Court order: witness statement required"),
    ("legal", "Employment Tribunal", "listing@employmenttribunals.gov.uk", "Tribunal hearing listed: preliminary"),
    ("legal", "First-tier Tribunal", "listing@tribunals.gov.uk", "Directions compliance required"),
    ("legal", "Admin Court", "orders@administrativecourt.gov.uk", "Judicial review: acknowledgment required"),
    ("legal", "Upper Tribunal", "listing@uppertribunal.gov.uk", "Permission hearing: skeleton argument"),
    ("legal", "Solicitor", "urgent@clientlaw.co.uk", "Injunction application: supporting evidence needed"),
    ("legal", "Bailiff Office", "enforcement@county-court.gov.uk", "Warrant of control: payment required"),
    ("legal", "SFO", "enquiries@sfo.gov.uk", "Document preservation notice"),
    ("legal", "Client Legal", "legal@manufacturing.co.uk", "Settlement offer: response required"),
    ("legal", "Coroner", "court@coroner.gov.uk", "Inquest: your attendance is required"),
    ("legal", "CPS", "witnesses@cps.gov.uk", "Expert witness attendance required"),
    ("legal", "Insurance Litigator", "claims@insure-law.co.uk", "Subrogated claim: disclosure list"),
    ("legal", "Kohus EE", "info@kohus.ee", "Court proceedings: document submission required"),
    ("legal", "District Judge", "clerk@familycourt.gov.uk", "Case management directions issued"),
    ("legal", "Mediator", "booking@cedr.com", "Mediation session confirmed: bundle deadline"),
    ("legal", "IP Solicitor", "tm@trademarklawyers.co.uk", "Trademark opposition: deadline approaching"),
    ("legal", "Data Subject", "request@individual.com", "Subject access request under GDPR"),
    ("legal", "Contract Lawyer", "contracts@commerciallaw.co.uk", "Draft SPA: comments required"),
    ("legal", "Arbitrator", "admin@lcia.org", "Arbitration hearing: procedural directions"),

    # ──── FINANCIAL (20) ────
    ("financial", "HMRC", "penalties@hmrc.gov.uk", "Tax penalty appeal: action required"),
    ("financial", "VAT Office", "vat@hmrc.gov.uk", "VAT return Q4 2025: filing deadline"),
    ("financial", "Pension Regulator", "compliance@tpr.gov.uk", "Auto-enrolment re-declaration required"),
    ("financial", "Bank", "compliance@businessbank.co.uk", "KYC refresh: updated documents required"),
    ("financial", "Creditor", "collections@supplier.com", "Account on hold: outstanding balance"),
    ("financial", "Client Finance", "accounts@retailco.com", "Invoice MC-INV-4421 overdue 30 days"),
    ("financial", "Hiscox Insurance", "renewals@hiscox.com", "Professional indemnity renewal: cover lapsing"),
    ("financial", "Accountant", "accounts@accountingfirm.co.uk", "Management accounts: review needed"),
    ("financial", "Xero", "notifications@xero.com", "Bank reconciliation: 12 unmatched items"),
    ("financial", "Stripe", "receipts@stripe.com", "Monthly settlement: GBP 4,200 processed"),
    ("financial", "AML Team", "compliance@bankpartner.com", "Enhanced due diligence: documents needed"),
    ("financial", "Council Tax", "business-rates@council.gov.uk", "Business rates appeal: evidence required"),
    ("financial", "Payroll Provider", "deadlines@payrollco.com", "Payroll submission: data required"),
    ("financial", "Revenue IE", "returns@revenue.ie", "Corporation tax return: filing reminder"),
    ("financial", "EMTA", "info@emta.ee", "Estonian tax declaration: quarterly submission"),
    ("financial", "Zurich Insurance", "liability@zurich.co.uk", "Directors liability renewal: premium quote"),
    ("financial", "Expense System", "noreply@expensify.com", "Expense report pending approval"),
    ("financial", "Auditor", "audit@grantthornton.co.uk", "Year-end audit: information request list"),
    ("financial", "Pension Provider", "admin@pensionscheme.co.uk", "Contribution schedule: monthly upload due"),
    ("financial", "Credit Control", "credit@mobicycle.consulting", "Aged debtor report: 3 invoices over 60 days"),

    # ──── REGULATORY (20) ────
    ("regulatory", "Companies House", "filing@companieshouse.gov.uk", "Annual confirmation statement due"),
    ("regulatory", "ICO", "registration@ico.org.uk", "Data protection registration renewal"),
    ("regulatory", "Environment Agency", "permits@environment-agency.gov.uk", "Environmental permit variation: consultation"),
    ("regulatory", "FCA", "registration@fca.org.uk", "FCA registration: annual attestation"),
    ("regulatory", "Ofcom", "licensing@ofcom.org.uk", "Broadcasting licence renewal"),
    ("regulatory", "Local Authority", "planning@localcouncil.gov.uk", "Planning application determination"),
    ("regulatory", "UKAS", "accreditation@ukas.com", "Accreditation surveillance audit scheduled"),
    ("regulatory", "Standards Body", "iso@bsigroup.com", "ISO 14001 recertification: audit date"),
    ("regulatory", "TTJA", "info@ttja.ee", "Estonian business registration: annual update"),
    ("regulatory", "CRO Ireland", "filings@cro.ie", "Annual return: filing deadline approaching"),
    ("regulatory", "Trading Standards", "enforcement@tradingstandards.gov.uk", "Product compliance: information request"),
    ("regulatory", "HMRC Customs", "customs@hmrc.gov.uk", "EORI number: renewal and verification"),
    ("regulatory", "Charity Commission", "returns@charitycommission.gov.uk", "Annual return: submission required"),
    ("regulatory", "DEFRA", "returns@defra.gov.uk", "Packaging waste: producer registration"),
    ("regulatory", "Planning Inspector", "hearings@pins.gov.uk", "Inquiry: statement of case required"),
    ("regulatory", "Health Board", "registration@cqc.org.uk", "Provider registration: conditions review"),
    ("regulatory", "Food Standards", "registration@food.gov.uk", "Food premises: registration renewal"),
    ("regulatory", "WRC Ireland", "compliance@workplacerelations.ie", "Employment law compliance inspection"),
    ("regulatory", "Gambling Commission", "licensing@gamblingcommission.gov.uk", "Operating licence: annual fee due"),
    ("regulatory", "Data Commission IE", "registration@dataprotection.ie", "DPC registration: renewal notice"),

    # ──── COMMERCIAL (20) ────
    ("commercial", "Sarah Chen", "sarah@greencorp.com", "ESG assessment quotation request"),
    ("commercial", "New Lead", "enquiry@techstartup.com", "Interested in carbon footprint assessment"),
    ("commercial", "Client Manager", "ops@foodmanufacturer.com", "Site visit to arrange for waste audit"),
    ("commercial", "Partner Firm", "collab@enviro-consultants.com", "Joint proposal for council tender"),
    ("commercial", "Client Request", "enquiries@energyco.com", "RFP for corporate sustainability strategy"),
    ("commercial", "Client Sustainability", "esg@bankinggroup.com", "Board ESG presentation: send deck draft"),
    ("commercial", "Subcontractor", "work@freelance-auditor.com", "Availability for site assessment?"),
    ("commercial", "Supplier", "orders@labequipment.com", "Monitoring equipment delivery: confirm address"),
    ("commercial", "Client Ops", "operations@logisticsgroup.com", "Supply chain mapping workshop: pick a date"),
    ("commercial", "Client Sustainability 2", "esg@retailchain.com", "Annual sustainability report: data collection"),
    ("commercial", "Procurement", "tenders@publicsector.gov.uk", "ITT for environmental consulting services"),
    ("commercial", "Film Client", "production@studiogroup.com", "Documentary commission: scope and pricing"),
    ("commercial", "Game Publisher", "biz@gamepublisher.com", "Educational game licensing discussion"),
    ("commercial", "Landlord", "property@commercialreits.com", "Rent review notice: response required"),
    ("commercial", "Service Provider", "contracts@cleaningco.com", "Office cleaning contract: review and renew"),
    ("commercial", "Charity Partner", "partnership@enviro-charity.org", "Corporate sponsorship renewal discussion"),
    ("commercial", "Fleet Manager", "vehicles@fleetco.com", "Company vehicle lease expiry: replacement options"),
    ("commercial", "IT Vendor", "renewals@softwareco.com", "Annual SaaS license renewal: review terms"),
    ("commercial", "Real Estate Agent", "commercial@estateagent.co.uk", "Office space options for expansion"),
    ("commercial", "Cloud Provider", "enterprise@cloudco.com", "Infrastructure contract review"),

    # ──── TECHNICAL (20) ────
    ("technical", "Cloudflare", "noreply@notify.cloudflare.com", "Workers deployment failed: error in worker script"),
    ("technical", "GitHub", "noreply@github.com", "Security advisory: critical vulnerability in dependency"),
    ("technical", "Cloudflare Status", "status@cloudflarestatus.com", "Incident: elevated error rates in Workers"),
    ("technical", "npm Security", "security@npmjs.com", "npm audit: 3 high severity vulnerabilities found"),
    ("technical", "SSL Monitor", "alerts@sslmate.com", "SSL certificate expiring in 14 days: mobicycle.eu"),
    ("technical", "Uptime Robot", "alerts@uptimerobot.com", "Monitor DOWN: cf-email.mobicycle.us (502)"),
    ("technical", "Google Cloud", "noreply@google.com", "API quota exceeded: Workspace API"),
    ("technical", "DNS Alert", "alerts@dnscheck.com", "DNSSEC validation failure detected"),
    ("technical", "GitHub Actions", "noreply@github.com", "Workflow run failed: deploy-workers"),
    ("technical", "Sentry", "noreply@sentry.io", "New issue: TypeError in cf-email handler"),
    ("technical", "Cloudflare R2", "noreply@cloudflare.com", "R2 bucket storage approaching limit"),
    ("technical", "Proton Mail", "support@protonmail.com", "Bridge connection: authentication required"),
    ("technical", "LetsEncrypt", "expiry@letsencrypt.org", "Certificate expiry notice"),
    ("technical", "Wrangler", "noreply@cloudflare.com", "Worker deployment successful: cf-dns"),
    ("technical", "GitHub Dependabot", "dependabot@github.com", "Bump hono from 4.5.0 to 4.6.2"),
    ("technical", "Docker Hub", "noreply@docker.com", "Image vulnerability scan: 2 critical CVEs"),
    ("technical", "PagerDuty", "alerts@pagerduty.com", "Incident triggered: high error rate on API"),
    ("technical", "Vercel", "noreply@vercel.com", "Build failed: node version incompatible"),
    ("technical", "AWS", "no-reply@sns.amazonaws.com", "CloudWatch alarm: CPU utilisation > 90%"),
    ("technical", "IT Support", "helpdesk@mobicycle.tech", "System migration: scheduled maintenance window"),

    # ──── ENVIRONMENTAL (20) ────
    ("environmental", "SBTi", "validation@sciencebasedtargets.org", "Target validation: additional data required"),
    ("environmental", "CDP", "disclosure@cdp.net", "CDP climate questionnaire: submission deadline"),
    ("environmental", "GRI", "updates@globalreporting.org", "Updated GRI 306 waste standard: review"),
    ("environmental", "Ellen MacArthur Foundation", "circular@emf.org", "Circular economy certification: next steps"),
    ("environmental", "DEFRA Emissions", "ghg@defra.gov.uk", "Updated UK emissions conversion factors 2026"),
    ("environmental", "Environment Agency", "waste@environment-agency.gov.uk", "Waste carrier licence: renewal due"),
    ("environmental", "CSRD Advisor", "csrd@sustainability-advisors.eu", "CSRD double materiality assessment: workshop"),
    ("environmental", "Carbon Trust", "certification@carbontrust.com", "Carbon footprint verification: evidence needed"),
    ("environmental", "Sustainalytics", "esg-ratings@sustainalytics.com", "ESG risk rating: data update request"),
    ("environmental", "MSCI ESG", "ratings@msci.com", "ESG rating review: response window open"),
    ("environmental", "TNFD", "disclosure@tnfd.global", "Nature-related disclosure: pilot programme"),
    ("environmental", "WEEE Compliance", "returns@weee-compliance.org", "WEEE producer registration: quarterly return"),
    ("environmental", "ISO 14001 Auditor", "audit@environmental-certification.com", "Surveillance audit: NC closeout evidence"),
    ("environmental", "EU Taxonomy", "taxonomy@eu-platform.europa.eu", "Taxonomy alignment: technical screening criteria update"),
    ("environmental", "Ecovadis", "assessment@ecovadis.com", "Sustainability assessment: questionnaire open"),
    ("environmental", "Climate Disclosure", "reporting@tcfd-hub.org", "TCFD implementation: scenario analysis guidance"),
    ("environmental", "Waste Broker", "collections@waste-management.co.uk", "Hazardous waste collection: consignment note"),
    ("environmental", "Biodiversity Net Gain", "bng@naturalengland.org.uk", "BNG assessment: baseline habitat survey results"),
    ("environmental", "Supply Chain ESG", "sustainability@suppliercorp.com", "Scope 3 data request: your emissions as our supplier"),
    ("environmental", "Energy Bureau", "certificates@ofgem.gov.uk", "REGO certificate: renewable energy guarantee"),

    # ──── HEALTH (20) ────
    ("health", "HSE", "investigations@hse.gov.uk", "Prohibition notice: cease operations pending review"),
    ("health", "HSE Inspections", "inspections@hse.gov.uk", "Workplace inspection findings: response required"),
    ("health", "RIDDOR", "reporting@riddor.gov.uk", "RIDDOR report confirmation: ref RID-2026-1847"),
    ("health", "Occupational Health", "referrals@ohprovider.co.uk", "Fitness for work assessment: appointment"),
    ("health", "Fire Safety", "inspections@fire-service.gov.uk", "Fire risk assessment: remediation required"),
    ("health", "MHRA", "safety@mhra.gov.uk", "Product safety alert: medical device recall"),
    ("health", "First Aid Trainer", "bookings@firstaid-training.co.uk", "First aid certificate renewal: course booked"),
    ("health", "COSHH Assessor", "assessments@chemicalsafety.co.uk", "COSHH assessment update: new substance added"),
    ("health", "EAP Provider", "support@eap-wellbeing.com", "Employee assistance programme: monthly report"),
    ("health", "DSE Assessor", "ergonomics@workplacehealth.co.uk", "DSE assessment: workstation adjustments needed"),
    ("health", "PAT Testing", "certificates@pat-testing.co.uk", "Portable appliance test: 4 items failed"),
    ("health", "Asbestos Surveyor", "reports@asbestos-management.co.uk", "Asbestos management survey: report attached"),
    ("health", "PPE Supplier", "orders@safety-equipment.co.uk", "PPE order confirmation: hard hats and hi-vis"),
    ("health", "Mental Health First Aid", "courses@mhfa.org.uk", "Mental health first aider training: next cohort"),
    ("health", "OSHA", "alerts@osha.gov", "Workplace safety citation: response required"),
    ("health", "Legionella Testing", "results@watertesting.co.uk", "Legionella risk assessment: results attached"),
    ("health", "Safety Committee", "h&s@mobicycle.consulting", "Monthly H&S committee meeting: agenda"),
    ("health", "Tööinspektsioon", "kontroll@ti.ee", "Estonian labour inspection: findings report"),
    ("health", "Near Miss Report", "safety@mobicycle.us", "Near miss: slippery floor incident warehouse B"),
    ("health", "Noise Assessment", "reports@acoustics-consulting.co.uk", "Noise at work assessment: action levels exceeded"),

    # ──── ADMINISTRATIVE (20) ────
    ("administrative", "Internal HR", "hr@mobicycle.consulting", "Performance reviews: complete self-assessment"),
    ("administrative", "Recruitment", "jobs@talentfirm.com", "Interview scheduling: senior consultant candidates"),
    ("administrative", "BambooHR", "noreply@bamboohr.com", "Leave request approved: 5-9 April"),
    ("administrative", "Office Manager", "facilities@mobicycle.consulting", "Office refurbishment planning: input needed"),
    ("administrative", "Training Provider", "courses@enviro-training.com", "Environmental auditor certification: next cohort"),
    ("administrative", "Internal Social", "fun@mobicycle.consulting", "Summer team social: vote for activity"),
    ("administrative", "Travel Agent", "bookings@corporatetravel.com", "Flight booking confirmation: LHR-TLL 15 April"),
    ("administrative", "Onboarding", "welcome@mobicycle.us", "New starter checklist: equipment and access"),
    ("administrative", "IT Admin", "admin@mobicycle.tech", "Password reset: your account was locked"),
    ("administrative", "Workday", "noreply@workday.com", "Timesheet submission reminder: week ending 21 March"),
    ("administrative", "CIEEM", "membership@cieem.net", "Professional membership renewal: payment due"),
    ("administrative", "CPD Portal", "cpd@rics.org", "CPD requirements: 12 hours remaining this year"),
    ("administrative", "Meeting Scheduler", "calendar@mobicycle.consulting", "All-hands meeting: Wednesday 10am"),
    ("administrative", "Company Secretary", "secretariat@mobicycle.us", "Board papers: review before Thursday"),
    ("administrative", "Probation Review", "hr@mobicycle.eu", "Probation review meeting: scheduled for Friday"),
    ("administrative", "Exit Interview", "hr@mobicycle.productions", "Exit interview: please complete feedback form"),
    ("administrative", "DBS Check", "screening@disclosure-service.gov.uk", "Enhanced DBS check: result available"),
    ("administrative", "Sabbatical Policy", "hr@mobicycle.consulting", "Updated sabbatical policy: please acknowledge"),
    ("administrative", "Stationery", "orders@officestuff.com", "Office supply order confirmed: delivery Thursday"),
    ("administrative", "Reception", "reception@mobicycle.consulting", "Visitor badge: client arriving 2pm"),
]

# Generate train-type.csv
random.shuffle(EXAMPLES)
os.makedirs('/content/data/type', exist_ok=True)
train_path = '/content/data/type/train-type.csv'
with open(train_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['text'])
    for label, name, email, subject in EXAMPLES:
        prompt = (f'<s>[INST] Classify this email\'s type as one of: legal, financial, '
                  f'regulatory, commercial, technical, environmental, health, administrative.\n\n'
                  f'To: {ACCOUNT.replace(".", " ").title()} <enquiries@{ACCOUNT}>\n'
                  f'From: {name} <{email}>\n'
                  f'Subject: {subject} [/INST] {label}</s>')
        w.writerow([prompt])

# Verify distribution
from collections import Counter
counts = Counter(label for label, *_ in EXAMPLES)
print(f'Total: {len(EXAMPLES)} examples')
for label, count in sorted(counts.items()):
    print(f'  {label}: {count} ({count/len(EXAMPLES)*100:.0f}%)')
print(f'Saved to {train_path}')


In [ ]:
# ── Cell 4: Train QLoRA ──
import sys, types, json, os, math

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))
    print('Config reloaded')

# Fix Colab triton bug
try:
    import triton
    if not hasattr(triton, 'ops'):
        triton.ops = types.ModuleType('triton.ops')
        sys.modules['triton.ops'] = triton.ops
except ImportError:
    pass

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
import torch

project = f'type-{ACCOUNT}'
data_dir = '/content/data/type'
output_dir = f'/content/{project}'

print(f'Training: {project}')

# Load dataset
ds = load_dataset('csv', data_files=f'{data_dir}/train-type.csv', split='train')
print(f'Loaded {len(ds)} examples')

steps_per_epoch = math.ceil(len(ds) / (BATCH_SIZE * GRADIENT_ACCUMULATION))
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_steps * 0.1)

# 4-bit quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = BLOCK_SIZE
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map={'': 0},
    torch_dtype=torch.float16, trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
print(f'Model loaded: {MODEL} (4-bit)')

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    gradient_checkpointing=True,
    max_grad_norm=0.0,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy='epoch',
    report_to='none',
)

trainer = SFTTrainer(
    model=model, train_dataset=ds, args=training_args,
    peft_config=lora_config, processing_class=tokenizer,
)

print('Starting training...')
trainer.train()

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'Done! Adapter saved to {output_dir}')

del model, trainer, tokenizer
torch.cuda.empty_cache()


In [ ]:
# ── Cell 5: Patch adapter_config.json for Workers AI ──
import json, os

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

project = f'type-{ACCOUNT}'
output_dir = f'/content/{project}'
config_path = os.path.join(output_dir, 'adapter_config.json')

with open(config_path) as f:
    config = json.load(f)
config['model_type'] = MODEL_TYPE
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

for name in ['adapter_model.safetensors', 'adapter_config.json']:
    path = os.path.join(output_dir, name)
    if os.path.exists(path):
        print(f'{name}: {os.path.getsize(path) / 1e6:.1f} MB')
    else:
        print(f'{name}: MISSING!')
print(f'r={config.get("r")}, alpha={config.get("lora_alpha")}, model_type={config.get("model_type")}')


In [ ]:
# ── Cell 6: Save Locally (Base64 extraction) ──
# MANDATORY: Save adapter files BEFORE deploying.
import base64, os, json

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

project = f'type-{ACCOUNT}'
output_dir = f'/content/{project}'

for name in ['adapter_model.safetensors', 'adapter_config.json']:
    path = os.path.join(output_dir, name)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            data = f.read()
        b64 = base64.b64encode(data).decode('ascii')
        print(f'--- {name} ({len(data)} bytes) ---')
        print(f'Save to: models/fine-tuning/PEFT/LoRA/type/{ACCOUNT}/adapter/{name}')
        print(f'Base64 length: {len(b64)}')
    else:
        print(f'MISSING: {path}')

print(f'\nDeploy to: models/fine-tuning/PEFT/LoRA/type/{ACCOUNT}/adapter/')


In [ ]:
# ── Cell 7: Deploy to Workers AI via cf-adapters ──
import requests, json, os, time

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

project = f'type-{ACCOUNT}'
output_dir = f'/content/{project}'

# cf-adapters worker URLs per account
ADAPTERS_URLS = {
    'mobicycle.us': 'https://adapters.mobicycle.workers.dev',
    'mobicycle.consulting': 'https://adapters.mobicycle-consulting.workers.dev',
    'mobicycle.eu': 'https://adapters.mobicycle-eu.workers.dev',
    'mobicycle.ee': 'https://adapters.mobicycle-ou.workers.dev',
    'mobicycle.productions': 'https://adapters.mobicycle-productions.workers.dev',
    'mobicycle.games': 'https://adapters.mobicycle-games.workers.dev',
    'mobicycle.tech': 'https://adapters.mobicycle-tech.workers.dev',
    'mobicycle.marketing': 'https://adapters.mobicycle-marketing.workers.dev',
    'mobicycle.capital': 'https://adapters.mobicycle-capital.workers.dev',
    'mobicycle.support': 'https://adapters.mobicycle-support.workers.dev',
    'mobicycle.news': 'https://adapters.mobicycle-news.workers.dev',
}

base_url = ADAPTERS_URLS.get(ACCOUNT)
if not base_url:
    raise ValueError(f'No adapters URL for {ACCOUNT}')

adapter_name = project
print(f'Deploying {adapter_name} to {base_url}')

# Step 1: Check current state
status = requests.get(f'{base_url}/api/status').json()
print(f'Current adapters: {json.dumps(status, indent=2)[:500]}')

# Step 2: Delete existing type entry if present
adapters = requests.get(f'{base_url}/api/adapters').json()
for a in adapters:
    if a.get('name') == adapter_name or a.get('description', '').startswith('Stage 1 type'):
        print(f'Deleting existing entry: {a["id"]}')
        requests.delete(f'{base_url}/api/adapters/{a["id"]}')
        time.sleep(2)

# Step 3: Create new finetune entry
create_resp = requests.post(f'{base_url}/api/adapters', json={
    'name': adapter_name,
    'model': WORKERS_AI_MODEL,
    'description': f'Stage 1 type classification LoRA for {ACCOUNT}'
})
create_data = create_resp.json()
finetune_id = create_data.get('id') or create_data.get('finetune', {}).get('id')
print(f'Created finetune: {finetune_id}')

# Step 4: Upload adapter files
for filename in ['adapter_model.safetensors', 'adapter_config.json']:
    filepath = os.path.join(output_dir, filename)
    with open(filepath, 'rb') as f:
        upload_resp = requests.post(
            f'{base_url}/api/adapters/{finetune_id}/upload',
            files={'file': (filename, f, 'application/octet-stream')},
            data={'file_name': filename}
        )
    print(f'Uploaded {filename}: {upload_resp.status_code}')

print(f'\nDeployment complete. Wait 5s then verify...')
time.sleep(5)


In [ ]:
# ── Cell 8: Verify Adapter Deployed ──
import requests, json

if 'ACCOUNT' not in dir():
    with open('/content/config.json') as f:
        globals().update(json.load(f))

# cf-adapters worker URLs per account
ADAPTERS_URLS = {
    'mobicycle.us': 'https://adapters.mobicycle.workers.dev',
    'mobicycle.consulting': 'https://adapters.mobicycle-consulting.workers.dev',
    'mobicycle.eu': 'https://adapters.mobicycle-eu.workers.dev',
    'mobicycle.ee': 'https://adapters.mobicycle-ou.workers.dev',
    'mobicycle.productions': 'https://adapters.mobicycle-productions.workers.dev',
    'mobicycle.games': 'https://adapters.mobicycle-games.workers.dev',
    'mobicycle.tech': 'https://adapters.mobicycle-tech.workers.dev',
    'mobicycle.marketing': 'https://adapters.mobicycle-marketing.workers.dev',
    'mobicycle.capital': 'https://adapters.mobicycle-capital.workers.dev',
    'mobicycle.support': 'https://adapters.mobicycle-support.workers.dev',
    'mobicycle.news': 'https://adapters.mobicycle-news.workers.dev',
}

base_url = ADAPTERS_URLS.get(ACCOUNT)
if not base_url:
    raise ValueError(f'No adapters URL for {ACCOUNT}')

adapter_name = f'type-{ACCOUNT}'
print(f'Verifying {adapter_name} on {base_url}...')

# Check adapter appears in status
status = requests.get(f'{base_url}/api/status').json()
adapters = requests.get(f'{base_url}/api/adapters').json()

found = False
for a in adapters:
    if a.get('name') == adapter_name:
        found = True
        print(f'FOUND: {a["name"]}')
        print(f'  ID: {a.get("id")}')
        print(f'  Model: {a.get("model")}')
        print(f'  Description: {a.get("description")}')
        break

if not found:
    print(f'NOT FOUND: {adapter_name}')
    print(f'Available adapters:')
    for a in adapters:
        print(f'  - {a.get("name")} ({a.get("model")})')
else:
    print(f'\nAdapter deployed. Ready for use with:')
    print(f'  model: {WORKERS_AI_MODEL}')
    print(f'  lora: {adapter_name}')